In [2]:
import torch
import torch.nn as nn

In [95]:
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction_rate=2):
        super(ChannelAttention, self).__init__()

        self.squeeze = nn.ModuleList([
            nn.AdaptiveAvgPool2d(1),
            nn.AdaptiveMaxPool2d(1)
        ])

        self.excitation = nn.Sequential(
            nn.Conv2d(in_channels=channels,
                      out_channels=channels // reduction_rate,
                      kernel_size=1,
                      bias=False),
            nn.SiLU(),
            nn.Conv2d(in_channels=channels // reduction_rate,
                      out_channels=channels,
                      kernel_size=1,
                      bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # perform squeeze with independent Pooling
        avg_feat = self.squeeze[0](x)
        max_feat = self.squeeze[1](x)
        # perform excitation with the same excitation sub-net
        avg_out = self.excitation(avg_feat)
        max_out = self.excitation(max_feat)
        # attention
        attention = self.sigmoid(avg_out + max_out)
        return attention * x
ca = ChannelAttention(4)

In [96]:
image = torch.rand(1, 3, 8, 8)
image = nn.Conv2d(3, 4, 3, bias=False)(image)

In [97]:
image

tensor([[[[ 0.1743,  0.4497,  0.2306,  0.0794,  0.2120,  0.2618],
          [ 0.0098,  0.3278,  0.0760, -0.1437,  0.0036,  0.1107],
          [ 0.4554,  0.4144,  0.5569,  0.2362,  0.1813,  0.5071],
          [ 0.1330,  0.0606,  0.5583,  0.2684,  0.3132,  0.4244],
          [ 0.0226,  0.3602,  0.5880,  0.2602,  0.4356, -0.0477],
          [ 0.0755,  0.1488,  0.2007,  0.3472,  0.0525, -0.0714]],

         [[ 0.1173,  0.0611, -0.0572,  0.1300,  0.2964, -0.0119],
          [-0.0487, -0.0261,  0.2986, -0.0472, -0.3953, -0.2145],
          [ 0.1423, -0.1430, -0.2215,  0.3059,  0.1553,  0.0091],
          [ 0.1110, -0.3527, -0.2324, -0.1796,  0.1107, -0.0077],
          [-0.3372, -0.1640, -0.2578,  0.2496, -0.1605, -0.1226],
          [ 0.1112, -0.1565, -0.0330, -0.1592, -0.0078, -0.2159]],

         [[ 0.2817,  0.1103,  0.3790,  0.4191,  0.1823, -0.0067],
          [ 0.1058,  0.3466,  0.3321,  0.3729,  0.2727,  0.4306],
          [ 0.3669,  0.3703,  0.4289,  0.2791,  0.1798,  0.1409],
      

In [98]:
a = ca(image)

In [1]:
3394848 - 3332496

62352

In [ ]:
nn.LayerNorm()(a)

In [101]:
nn.functional.layer_norm(a, a.size()[0:])

tensor([[[[ 0.4920,  1.4325,  0.6842,  0.1682,  0.6208,  0.7907],
          [-0.0696,  1.0163,  0.1565, -0.5936, -0.0907,  0.2751],
          [ 1.4518,  1.3118,  1.7985,  0.7036,  0.5159,  1.6284],
          [ 0.3511,  0.1041,  1.8033,  0.8135,  0.9664,  1.3461],
          [-0.0256,  1.1267,  1.9045,  0.7852,  1.3843, -0.2657],
          [ 0.1548,  0.4050,  0.5823,  1.0823,  0.0763, -0.3466]],

         [[ 0.3021,  0.1078, -0.3002,  0.3457,  0.9202, -0.1442],
          [-0.2711, -0.1929,  0.9277, -0.2660, -1.4674, -0.8433],
          [ 0.3882, -0.5966, -0.8675,  0.9531,  0.4332, -0.0715],
          [ 0.2803, -1.3204, -0.9050, -0.7231,  0.2793, -0.1294],
          [-1.2670, -0.6689, -0.9929,  0.7586, -0.6571, -0.5262],
          [ 0.2810, -0.6433, -0.2170, -0.6525, -0.1300, -0.8482]],

         [[ 0.8110,  0.2549,  1.1267,  1.2567,  0.4884, -0.1247],
          [ 0.2403,  1.0217,  0.9746,  1.1068,  0.7819,  1.2940],
          [ 1.0875,  1.0985,  1.2887,  0.8025,  0.4805,  0.3542],
      